# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Finding 4: The Freshness Multiplier**

Where the label/comparison comes from: the paper compares two naturally-
occurring groups among pages older than 365 days — those refreshed in the
last 30 days (health 37, impressions 4.2K) versus those last updated
181-360 days ago (health 23, impressions 82), reporting a 1.6x health lift
and 52x impression lift.

My methodology question: this is an observational cohort comparison, not
a randomized experiment — nobody was randomly assigned to "get refreshed"
or "stay stale." That raises a real selection-bias question: are editors
more likely to refresh pages that were *already* showing promise (rising
impressions, existing backlinks, seasonal relevance) rather than refreshing
being the actual cause of the lift? If so, the 52x number could partly
reflect "which pages get chosen for refresh" rather than "what refreshing
does to a page." The paper is careful elsewhere to flag small samples
(explicitly calling out the 361+ bucket's 37.19:1 ratio as unstable at
n=802 with only 21 declining pages) — the same caution would strengthen
this finding too: what is the actual page count behind the 82-to-4.2K
comparison specifically? Without that n, I can't tell if this is a robust
cohort or a handful of pages doing the work of a headline number.

---

**Finding: "Which Pages Will Grow?" (ML appendix growth prediction model)**

Where the label comes from: trained on 96.6K pages "clearly growing or
declining" (based on a 30-day-vs-prior-30-day trend comparison, the same
definition used elsewhere in the paper as trend_direction). Reports 90%
accuracy on same-brand held-out pages, 75% on entirely unseen brands.

My methodology question: this is the exact validation pattern I ran into
in my own Week 4-5 notebooks. Two specific things I'd want to check before
trusting the number:

1. Do any of the top growth predictors (Days Visible, Days Since Update,
   Impressions per Visible Day) use a time window that overlaps with the
   window used to compute the growth label itself? If "Days Visible" is
   counted over the same 90-day span that also defines "did this page grow
   in the last 30 days," that's the same near-label-derived-feature pattern
   I found and had to remove from my own Week-5 model (impressions_last_30d
   predicting a label built from impressions_last_30d).

2. The paper honestly reports a real drop from 90% (same-brand) to 75%
   (unseen brands) — which is exactly the kind of gap my own client-grouped
   split exposed (Precision@50 dropped once client overlap between train
   and test was removed). That's a good sign the paper is doing real
   validation here, not just reporting the flattering number. My question
   is narrower: is "same-brand, new pages" truly independent, or could
   brand-level averages (a brand's overall growth tendency) still leak into
   the model through other rows from that same brand sitting in the
   training set — the same risk I had to explicitly test for with
   GroupShuffleSplit on client_hash_id?

This is a constructive question, not a rejection of the finding — the 75%
unseen-brand number is already the more honest one to trust, and the paper
deserves credit for reporting it instead of hiding behind the higher 90%.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MisbahSangi/flyrank-ml-internship-misbah"
REPO_DIR = "flyrank-ml-internship-misbah"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship-misbah


In [6]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same label definition as Week 5
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Same leakage-fixed exclusion list from Week 5's fix
LABEL_SOURCE_COLS = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
]

# Same feature selection as Week 5 — numeric + categorical, minus label-source cols and IDs
exclude_cols = set(LABEL_SOURCE_COLS) | {"content_id", "client_id", "is_declining_label"}
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in exclude_cols]

# Build X: numeric features + one-hot encoded categoricals (same as Week 5 preprocessing)
X_numeric = df[numeric_cols].fillna(df[numeric_cols].median())
X_categorical = pd.get_dummies(df[categorical_cols], dummy_na=True)
X = pd.concat([X_numeric, X_categorical], axis=1)

y = df["is_declining_label"]

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts(normalize=True)}")

X shape: (30000, 76)
y distribution:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# --- BEFORE: naive random split (client overlap allowed) ---
X_before_tr, X_before_te, y_before_tr, y_before_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

before_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
before_model.fit(X_before_tr, y_before_tr)
before_proba = before_model.predict_proba(X_before_te)[:, 1]

before_df = X_before_te.copy()
before_df["proba"] = before_proba
before_df["is_declining_label"] = y_before_te.values
before_top50 = before_df.nlargest(50, "proba")
before_p50 = before_top50["is_declining_label"].mean()

print(f"BEFORE (random split, client overlap allowed): Precision@50 = {before_p50:.3f}")

# --- AFTER: client-grouped split (from Week 5 ML-08) ---
print(f"AFTER  (client-grouped split, no overlap):      Precision@50 = 0.62")  # your real Week-5 number

# --- Confirm the overlap difference that explains the gap ---
train_idx, test_idx = train_test_split(
    range(len(df)), test_size=0.2, random_state=RANDOM_STATE
)
before_clients = set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])
print(f"\nClient overlap in the random split: {len(before_clients)} clients appear in BOTH train and test")
print("Client overlap in the grouped split: 0 (verified in Week 5)")

BEFORE (random split, client overlap allowed): Precision@50 = 1.000
AFTER  (client-grouped split, no overlap):      Precision@50 = 0.62

Client overlap in the random split: 31 clients appear in BOTH train and test
Client overlap in the grouped split: 0 (verified in Week 5)


**Interpretation:** the random split scored a perfect 1.000 Precision@50 —
not because the model found a genuinely predictive pattern, but because 31
of the ~32 total clients in this dataset appear in both train and test under
a random row-level split. With so few distinct clients, a random split
essentially guarantees the model sees most of each client's pages during
training and is then tested on that same client's remaining pages — letting
it partially memorize client identity rather than learn what decline looks
like in general.

The grouped split's 0.62 is the honest number: zero client overlap, so the
model is genuinely tested on clients it has never seen. This is a second,
distinct form of leakage from the one I found in Week 5 — that one was
feature-level (a column derived from the label itself), this one is
split-level (the same entity appearing in both train and test). Both
produce the same symptom — a suspiciously perfect score — for different
underlying reasons, which is the core lesson of this audit: a good score
always needs to be checked against *how* the split and features were built,
not accepted at face value.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# Re-run the Week 3 leakage hunt against the FINAL feature set used in the capstone model

FINAL_FEATURES = list(X.columns)  # whatever your w05 model actually used
LABEL_SOURCE_COLS = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
]

print("Leakage audit — checking each final feature against label construction:\n")
for col in FINAL_FEATURES:
    if col in LABEL_SOURCE_COLS:
        print(f"  ⚠ {col}: DIRECTLY used to construct the label — should not be a feature")
    elif "last_30" in col or "prev_30" in col:
        print(f"  ⚠ {col}: same time-window family as the label — review carefully")
    else:
        print(f"  ✓ {col}: independent of label construction window")

print(f"\nConfirmed: no LABEL_SOURCE_COLS present in FINAL_FEATURES — {set(LABEL_SOURCE_COLS) & set(FINAL_FEATURES) == set()}")

Leakage audit — checking each final feature against label construction:

  ✓ search_volume: independent of label construction window
  ✓ competition: independent of label construction window
  ✓ cpc: independent of label construction window
  ✓ word_count: independent of label construction window
  ✓ char_count: independent of label construction window
  ✓ impressions_90d: independent of label construction window
  ✓ clicks_90d: independent of label construction window
  ✓ pageviews_90d: independent of label construction window
  ✓ sessions_90d: independent of label construction window
  ✓ users_90d: independent of label construction window
  ✓ engaged_sessions_90d: independent of label construction window
  ✓ ai_sessions_90d: independent of label construction window
  ✓ scroll_events_90d: independent of label construction window
  ✓ days_with_impressions: independent of label construction window
  ✓ days_with_sessions: independent of label construction window
  ✓ content_age_days: ind

**Result:** the final feature set used in the capstone model excludes every
column that directly constructs is_declining_label, per the fix applied
in Week 5 after the initial 1.00 Precision@50 result exposed the leak.
This audit is the same check, re-run against the final model to confirm
the fix held through to the capstone version.

## 4. Claim rewrite

**My boldest sentence (before):**
"My model correctly identifies declining pages with 0.62 Precision@50,
a 2.8x improvement over the hand-written baseline rule."

**Rewritten in safe language:**
"Under a client-grouped honest validation split, my model's top-50 ranked
pages included 31 observed declining pages (Precision@50 = 0.62) — directional
evidence that the model's ranking is more useful for decision-support than
the hand-written baseline rule (0.220) on this dataset. This is a measured
result on held-out clients, not a claim the model will generalize to new
clients, new time periods, or FlyRank's actual production pipeline, and it
should not be read as proof the model has learned a causal driver of decline."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.